In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch.optim as optim

# Configurations
DATA_PATH = "dataset/emotion_recognitions_merged.csv"
TEXT_COL = "text"
LABEL_COL = "label"
EPOCHS = 10
BATCH_SIZE = 64
LR = 1e-3
MAX_FEATURES = 1000

# Dataset Class
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# BoW Classifier Model
class BoWClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=6):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# train function
def train_model(model, loader, criterion, optimizer):
    model.train()
    for X, y in loader:
        optimizer.zero_grad()
        outputs = model(X.float())
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

# evaluate function
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            out = model(X.float())
            pred = torch.argmax(out, dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(y.cpu().numpy())
    print(classification_report(trues, preds))
    print(confusion_matrix(trues, preds))

if __name__ == "__main__":
    df = pd.read_csv(DATA_PATH)
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(int).tolist()

    X_train, X_temp, y_train, y_temp = train_test_split(texts, labels, test_size=0.3, stratify=labels)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)

    vectorizer = CountVectorizer(max_features=MAX_FEATURES)
    X_train_bow = vectorizer.fit_transform(X_train).toarray()
    X_val_bow = vectorizer.transform(X_val).toarray()
    X_test_bow = vectorizer.transform(X_test).toarray()

    train_ds = TextDataset(torch.tensor(X_train_bow), torch.tensor(y_train))
    val_ds = TextDataset(torch.tensor(X_val_bow), torch.tensor(y_val))
    test_ds = TextDataset(torch.tensor(X_test_bow), torch.tensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    model = BoWClassifier(input_dim=MAX_FEATURES)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        train_model(model, train_loader, criterion, optimizer)
        print(f"Epoch {epoch+1} validation:")
        evaluate(model, val_loader)

    print("Final Test Performance:")
    evaluate(model, test_loader)


Epoch 1 validation:
              precision    recall  f1-score   support

           0       0.88      0.92      0.90     19047
           1       0.86      0.92      0.89     22174
           2       0.89      0.63      0.74      5429
           3       0.86      0.79      0.82      9004
           4       0.84      0.77      0.80      7513
           5       0.69      0.82      0.75      2354

    accuracy                           0.86     65521
   macro avg       0.84      0.81      0.82     65521
weighted avg       0.86      0.86      0.85     65521

[[17523   701    48   443   270    62]
 [  718 20406   295   321   200   234]
 [  227  1621  3430    84    44    23]
 [  883   563    43  7087   414    14]
 [  603   357    16   242  5766   529]
 [   69   154     6    24   174  1927]]
Epoch 2 validation:
              precision    recall  f1-score   support

           0       0.92      0.88      0.90     19047
           1       0.85      0.93      0.89     22174
           2       

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import torch.optim as optim

# Configurations
DATA_PATH = "dataset/emotion_recognitions_merged.csv"
TEXT_COL = "text"
LABEL_COL = "label"
EPOCHS = 10
BATCH_SIZE = 64
LR = 1e-3
MAX_FEATURES = 1000

# Dataset Class
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# BoW Classifier Model
class BoWClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=6):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# Train & Evaluate
def train_model(model, loader, criterion, optimizer):
    model.train()
    for X, y in loader:
        optimizer.zero_grad()
        outputs = model(X.float())
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            out = model(X.float())
            pred = torch.argmax(out, dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(y.cpu().numpy())
    print(classification_report(trues, preds))
    print(confusion_matrix(trues, preds))

# Main
if __name__ == "__main__":
    # 데이터 불러오기
    df = pd.read_csv(DATA_PATH)
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(int).tolist()

    # 데이터 분할
    X_train, X_temp, y_train, y_temp = train_test_split(
        texts, labels, test_size=0.3, stratify=labels, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )

    # BoW 벡터화
    vectorizer = CountVectorizer(max_features=MAX_FEATURES)
    X_train_bow = vectorizer.fit_transform(X_train).toarray()
    X_val_bow = vectorizer.transform(X_val).toarray()
    X_test_bow = vectorizer.transform(X_test).toarray()

    # Dataset & DataLoader
    train_ds = TextDataset(torch.tensor(X_train_bow), torch.tensor(y_train))
    val_ds = TextDataset(torch.tensor(X_val_bow), torch.tensor(y_val))
    test_ds = TextDataset(torch.tensor(X_test_bow), torch.tensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    # 모델, 클래스 가중치 계산
    model = BoWClassifier(input_dim=MAX_FEATURES)
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # 학습
    for epoch in range(EPOCHS):
        train_model(model, train_loader, criterion, optimizer)
        print(f"\nEpoch {epoch+1} Validation:")
        evaluate(model, val_loader)

    # 최종 테스트 성능
    print("\nFinal Test Performance:")
    evaluate(model, test_loader)



Epoch 1 Validation:
              precision    recall  f1-score   support

           0       0.95      0.84      0.90     19047
           1       0.96      0.81      0.88     22174
           2       0.64      0.92      0.76      5429
           3       0.73      0.87      0.79      9004
           4       0.78      0.79      0.79      7513
           5       0.59      0.94      0.73      2354

    accuracy                           0.84     65521
   macro avg       0.78      0.86      0.81     65521
weighted avg       0.86      0.84      0.85     65521

[[16090   391   351  1307   684   224]
 [  382 17973  2113   903   372   431]
 [   54   108  4991   198    45    33]
 [  201   166   200  7821   526    90]
 [  135   135   107   459  5953   724]
 [   11    27    17    60    36  2203]]

Epoch 2 Validation:
              precision    recall  f1-score   support

           0       0.95      0.86      0.90     19047
           1       0.95      0.82      0.88     22174
           2     